# VWAP Execution & the Almgren–Chriss Impact Model

**Where Alpha Dies on Its Way to the Market**

*Your backtest shows 15% returns. You deploy. Six months later, you're at 4%. What happened? Execution costs.*

This notebook builds a **production-grade execution engine** that:
- Implements a realistic **VWAP** (Volume-Weighted Average Price) algorithm
- Models **market impact** using the full **Almgren–Chriss (2001)** framework
- Computes the **optimal trading trajectory** (closed-form hyperbolic sine solution)
- Quantifies exactly how much alpha is destroyed by slippage and impact
- Integrates directly with the `ConvexPortfolioOptimizer` from the previous notebook

**Why it matters:**  
Execution is the #1 reason retail quant strategies fail in live trading — and the first thing funds test in interviews. This is the notebook that turns a paper strategy into a deployable one.

### 1. Mathematical Foundation

#### Almgren–Chriss Optimal Execution (2001)

We liquidate (or acquire) $X_0$ shares over $N$ discrete time steps (e.g., 1-minute slices in a trading day).
**Market impact decomposition:**
- **Permanent impact** (linear in total volume): $\gamma \cdot x_k$ (affects all future trades)
- **Temporary impact** (linear in trading rate): $\eta \cdot \frac{x_k}{\tau}$ (only affects the current slice)

**Implementation Shortfall (IS) cost:**
$$
\text{IS} = \sum_{k=1}^N x_k \cdot \left( \tilde{S}_k - S_0 \right)
$$
where $\tilde{S}_k$ is the execution price = unaffected price + temporary + cumulative permanent.

**Objective (risk-averse trader):**
$$
\min_{\{x_k\}} \quad \mathbb{E}[\text{IS}] + \lambda \cdot \text{Var}(\text{IS})
$$

**Closed-form solution** (continuous-time limit, discretized):
$$
X_t^* = X_0 \frac{\sinh(\kappa (T - t))}{\sinh(\kappa T)}, \quad \kappa = \sqrt{\frac{\lambda \sigma^2}{\eta}}
$$
where:
- $\sigma$: volatility
- $\eta$: temporary impact coefficient
- $\lambda$: risk aversion
- $T$: total horizon (e.g., 1 trading day)

**VWAP benchmark:** Trade $x_k \propto V_k$ (volume in slice $k$).




### Step 1: Import libraries and dependencies

In [9]:
import numpy as np
import pandas as pd
import yfinance as yf
import cvxpy as cp
import vectorbt as vbt
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import warnings

from datetime import datetime
from sklearn.covariance import LedoitWolf
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

### Step 2: Defining the Execution Engine Class

In [2]:
class ExecutionEngine:
  """
  Production-grade execution engine combining:
    - VWAP (volume-weighted benchmark)
    - Almgren-Chriss optimal execution (closed-form + numerical)
  """
    
  def __init__(self, eta: float = 0.0001, gamma: float = 0.00005, sigma: float = 0.015, lambda_risk: float = 1e-6, n_slices: int = 390):  # 390 minutes in a trading day
    self.eta = eta          # temporary impact (per unit trading rate)
    self.gamma = gamma      # permanent impact (per share)
    self.sigma = sigma      # daily volatility (fraction)
    self.lambda_risk = lambda_risk
    self.n_slices = n_slices
    self.kappa = None
    
  def _compute_kappa(self):
    """Risk-adjusted decay parameter"""
    self.kappa = np.sqrt(self.lambda_risk * self.sigma**2 / self.eta)
    return self.kappa
    
  # ====================== VWAP STRATEGY ======================  
  def vwap_trajectory(self, X0: float, volume_profile: np.ndarray):
    """Trade proportionally to expected volume (classic VWAP)"""
    volume_profile = np.array(volume_profile)
    volume_profile /= volume_profile.sum()
    trades = X0 * volume_profile
    return trades
    
  # ====================== ALMGREN-CHRISS OPTIMAL TRAJECTORY ======================
  def almgren_chriss_trajectory(self, X0: float, T: float = 1.0):
    """Closed-form hyperbolic sine optimal trajectory (continuous → discrete)"""
    self._compute_kappa()
    t = np.linspace(0, T, self.n_slices + 1)
    holdings = X0 * np.sinh(self.kappa * (T - t)) / np.sinh(self.kappa * T)
    trades = np.diff(holdings) * -1  # positive = sell (or buy if X0 < 0)
    return holdings[:-1], trades
    
  # ====================== MARKET IMPACT SIMULATION ======================
  def simulate_execution(self, S0: float, X0: float, trades: np.ndarray, random_seed: int = 42, show_impact: bool = True):
    """Simulate price path + impact for any strategy"""
    np.random.seed(random_seed)
    dt = 1.0 / self.n_slices
    unaffected = np.cumsum(np.random.normal(0, self.sigma * np.sqrt(dt), self.n_slices))
    S_unaffected = S0 * np.exp(unaffected)
    
    executed_prices = np.zeros(self.n_slices)
    remaining = X0
    cumulative_permanent = 0.0
    cost = 0.0
    
    for i in range(self.n_slices):
      x = trades[i]                     # shares traded this slice
      if abs(x) < 1e-6:
        executed_prices[i] = S_unaffected[i]
        continue
      
      temp_impact = self.eta * (x / dt)  # trading rate
      exec_price = S_unaffected[i] + temp_impact + cumulative_permanent
      
      executed_prices[i] = exec_price
      cost += x * exec_price
      cumulative_permanent += self.gamma * x
      remaining -= x
    
    total_cost = cost
    arrival_cost = X0 * S0
    is_shortfall = total_cost - arrival_cost   # implementation shortfall
    
    if show_impact:
      self._plot_execution(S_unaffected, executed_prices, trades, X0)
    
    return {
      'executed_prices': executed_prices,
      'implementation_shortfall': is_shortfall,
      'average_slippage_per_share': is_shortfall / abs(X0),
      'total_shares_executed': abs(X0)
    }
    
  def _plot_execution(self, S_unaffected, executed_prices, trades, X0):
    fig = make_subplots(rows=2, cols=1, subplot_titles=(
      "Price Path + Impact (Unaffected vs Executed)",
      "Trading Trajectory vs VWAP"
    ))
    
    t = np.arange(len(S_unaffected))
    fig.add_trace(go.Scatter(x=t, y=S_unaffected, name="Unaffected Price", line=dict(color='gray')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=executed_prices, name="Executed Price", line=dict(color='red')), row=1, col=1)
    
    # Cumulative traded
    cum_traded = np.cumsum(trades)
    fig.add_trace(go.Scatter(x=t, y=cum_traded, name="Cumulative Executed", line=dict(color='blue')), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=np.linspace(0, X0, len(t)), name="Linear (TWAP)", line=dict(dash='dash')), row=2, col=1)
    
    fig.update_layout(height=700, template='plotly_white', title="Almgren-Chriss Execution Simulation")
    fig.show()

### Step 3. Realistic Intraday Volume Profile (U-Shaped)

In [3]:
# Synthetic U-shaped volume profile (realistic for equities)
def generate_volume_profile(n_slices=390):
  t = np.linspace(0, 1, n_slices)
  volume = np.sin(np.pi * t)**2 + 0.2 * np.exp(-((t-0.5)**2)/0.05)  # open + close spikes
  volume += 0.1 * np.random.normal(0, 0.05, n_slices)
  return np.maximum(volume, 0.01)

volume_profile = generate_volume_profile()

### Step 4: VWAP vs. Almgren-Chriss

In [5]:
engine = ExecutionEngine(eta=0.00015, gamma=0.00008, sigma=0.018, lambda_risk=1e-5)

S0 = 150.0
X0 = 500_000   # large order (typical for institutional rebalance)

# 1. VWAP strategy
vwap_trades = engine.vwap_trajectory(X0, volume_profile)

# 2. Optimal Almgren-Chriss strategy
holdings_ac, ac_trades = engine.almgren_chriss_trajectory(X0)

print(f"Total shares to execute: {X0:,}")

# Simulate both
print("\n=== VWAP Execution ===")
vwap_results = engine.simulate_execution(S0, X0, vwap_trades, random_seed=42, show_impact=False)
print(f"Implementation Shortfall: ${vwap_results['implementation_shortfall']:,.2f}")
print(f"Slippage per share     : ${vwap_results['average_slippage_per_share']:.4f}")

print("\n=== Almgren-Chriss Optimal Execution ===")
ac_results = engine.simulate_execution(S0, X0, ac_trades, random_seed=42, show_impact=True)
print(f"Implementation Shortfall: ${ac_results['implementation_shortfall']:,.2f}")
print(f"Slippage per share     : ${ac_results['average_slippage_per_share']:.4f}")

Total shares to execute: 500,000

=== VWAP Execution ===
Implementation Shortfall: $67,107,464.59
Slippage per share     : $134.2149

=== Almgren-Chriss Optimal Execution ===


Implementation Shortfall: $47,217,955.62
Slippage per share     : $94.4359


### Step 5: Integration with ConvexPortfolioOptimizer (Alpha Decay)

In [6]:
# Reuse the optimizer from previous notebook (or paste the class here)
# For demo we assume we have target weights and previous weights → turnover shares

# Example: Rebalancing a $10M portfolio
portfolio_value = 10_000_000
target_weights = np.array([0.15, 0.12, 0.10, 0.08, 0.10, 0.08, 0.10, 0.07, 0.10, 0.10])  # from earlier example
prev_weights = np.array([0.20, 0.10, 0.15, 0.05, 0.08, 0.12, 0.05, 0.10, 0.08, 0.07])

turnover = np.abs(target_weights - prev_weights).sum()
shares_to_trade = (portfolio_value * (target_weights - prev_weights) / 150.0).astype(int)  # assume $150/share avg

print(f"Portfolio turnover: {turnover:.1%} → {abs(shares_to_trade).sum():,} shares to trade")

# Apply execution to the largest order (for illustration)
largest_order_idx = np.argmax(np.abs(shares_to_trade))
X0_port = abs(shares_to_trade[largest_order_idx])

ac_port = engine.almgren_chriss_trajectory(X0_port)
port_results = engine.simulate_execution(150.0, X0_port, ac_port[1])

print(f"\nExecution cost on this rebalance slice: ${port_results['implementation_shortfall']:,.2f}")
print("→ This is the exact drag that turns 15% backtest → 4% live")

Portfolio turnover: 34.0% → 22,664 shares to trade



Execution cost on this rebalance slice: $400.36
→ This is the exact drag that turns 15% backtest → 4% live


### Step 6. Cost vs. Risk Aversion Frontier (Interactive)

In [7]:
lambdas = np.logspace(-7, -3, 20)
shortfalls = []

for lam in lambdas:
  engine.lambda_risk = lam
  _, trades = engine.almgren_chriss_trajectory(X0)
  res = engine.simulate_execution(S0, X0, trades, random_seed=42, show_impact=False)
  shortfalls.append(res['implementation_shortfall'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=lambdas, y=shortfalls, mode='lines+markers', name='Implementation Shortfall'))
fig.update_layout(title="Almgren-Chriss Efficient Frontier: Cost vs. Risk Aversion", xaxis_title="Risk Aversion λ", yaxis_title="Implementation Shortfall ($)", template='plotly_white', height=500)
fig.show()

### Step 7: Integrating it with the ConvexPortfolioOptimiser code

Our Convex Portfolio Optimiser class

In [19]:
class ConvexPortfolioOptimizer:
  """
  Production-grade mean-variance optimizer with real-world constraints.
  """
  def __init__(self, tickers: list, sector_map: dict = None):
    self.tickers = tickers
    self.n = len(tickers)
    self.sector_map = sector_map or {}
    self.sector_indices = self._build_sector_indices()
    self.returns = None
    self.mu = None
    self.cov_shrunk = None

  def _build_sector_indices(self):
    sector_indices = {}
    for sec in set(self.sector_map.values()):
      sector_indices[sec] = [i for i, t in enumerate(self.tickers) if self.sector_map.get(t) == sec]
    return sector_indices

  def fetch_data(self, start_date='2018-01-01', end_date=None):
    if end_date is None:
      end_date = datetime.now().strftime('%Y-%m-%d')
    self.prices = yf.download(self.tickers, start=start_date, end=end_date)['Close']
    self.returns = self.prices.pct_change().dropna()
    return self.returns

  def compute_moments(self, annualization_factor=252):
    if self.returns is None:
      raise ValueError("Run fetch_data() first.")
    self.mu = self.returns.mean() * annualization_factor
    lw = LedoitWolf()
    self.cov_shrunk = lw.fit(self.returns).covariance_ * annualization_factor
    return self.mu, self.cov_shrunk

  def optimize(self, risk_aversion=3.0, position_bounds=(0.01, 0.20), sector_caps=None):
    mu = self.mu.values
    Sigma = self.cov_shrunk
    w = cp.Variable(self.n)

    objective = cp.Minimize(cp.quad_form(w, Sigma) - risk_aversion * mu.T @ w)
    constraints = [cp.sum(w) == 1.0]
    l, u = position_bounds
    constraints.append(w >= l)
    constraints.append(w <= u)

    if sector_caps is not None:
      for sec, cap in sector_caps.items():
        if sec in self.sector_indices:
          idx = self.sector_indices[sec]
          constraints.append(cp.sum(w[idx]) <= cap)

    prob = cp.Problem(objective, constraints)
    try:
      prob.solve(solver=cp.MOSEK, verbose=False)
    except:
      prob.solve(solver=cp.ECOS, verbose=False)

    if prob.status not in ['optimal', 'optimal_inaccurate']:
      raise ValueError(f"Optimization failed: {prob.status}")
    return w.value

Our enhanced ExecutionEngine with visualisation comparison

In [20]:
class ExecutionEngineEnhanced:
  """
  VWAP + Almgren-Chriss execution with clear performance & cost comparison.
  """
  def __init__(self, eta=0.00015, gamma=0.00008, sigma=0.018, lambda_risk=1e-5, n_slices=390):
    self.eta = eta
    self.gamma = gamma
    self.sigma = sigma
    self.lambda_risk = lambda_risk
    self.n_slices = n_slices

  def _compute_kappa(self):
    return np.sqrt(self.lambda_risk * self.sigma**2 / self.eta)

  def vwap_trajectory(self, X0: float, volume_profile: np.ndarray):
    volume_profile = np.array(volume_profile) / np.array(volume_profile).sum()
    return X0 * volume_profile

  def almgren_chriss_trajectory(self, X0: float, T=1.0):
    kappa = self._compute_kappa()
    t = np.linspace(0, T, self.n_slices + 1)
    holdings = X0 * np.sinh(kappa * (T - t)) / np.sinh(kappa * T)
    trades = -np.diff(holdings)
    return trades

  def simulate_execution(self, S0: float, X0: float, trades: np.ndarray, strategy_name="Strategy", random_seed=42, show_plot=True):
    np.random.seed(random_seed)
    dt = 1.0 / self.n_slices
    unaffected = np.cumsum(np.random.normal(0, self.sigma * np.sqrt(dt), self.n_slices))
    S_unaffected = S0 * np.exp(unaffected)

    executed_prices = np.zeros(self.n_slices)
    cost = 0.0
    cumulative_permanent = 0.0

    for i in range(self.n_slices):
      x = trades[i]
      if abs(x) < 1e-6:
        executed_prices[i] = S_unaffected[i]
        continue
      temp_impact = self.eta * (x / dt)
      exec_price = S_unaffected[i] + temp_impact + cumulative_permanent
      executed_prices[i] = exec_price
      cost += x * exec_price
      cumulative_permanent += self.gamma * x

    is_shortfall = cost - X0 * S0
    avg_slippage = is_shortfall / abs(X0) if X0 != 0 else 0

    if show_plot:
      self._plot_execution_comparison(S_unaffected, executed_prices, trades, X0, strategy_name)

    return {
      'implementation_shortfall': is_shortfall,
      'average_slippage_per_share': avg_slippage
    }

  def _plot_execution_comparison(self, S_unaffected, executed_prices, trades, X0, strategy_name):
    fig = make_subplots(rows=2, cols=1, subplot_titles=(f"Price Path & Impact - {strategy_name}", "Cumulative Execution Trajectory"), vertical_spacing=0.15)
    t = np.arange(len(S_unaffected))
    fig.add_trace(go.Scatter(x=t, y=S_unaffected, name="Unaffected Price", line=dict(color='gray', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=executed_prices, name=f"Executed ({strategy_name})", line=dict(color='red')), row=1, col=1)

    cum_traded = np.cumsum(trades)
    fig.add_trace(go.Scatter(x=t, y=cum_traded, name=f"Cumulative {strategy_name}", line=dict(color='blue')), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=np.linspace(0, X0, len(t)), name="Linear Benchmark", line=dict(dash='dot')), row=2, col=1)

    fig.update_layout(height=750, template='plotly_white', title=f"Execution Comparison - {strategy_name} vs Market Impact")
    fig.show()

Performance & Execution Comparison

In [41]:
# ========================== UNIVERSE ==========================
tickers = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'JPM', 'BAC', 'XOM', 'CVX', 'PG', 'KO']

sector_map = {
  t: 'Technology' if t in ['AAPL','MSFT','GOOGL','NVDA'] else
  'Financials' if t in ['JPM','BAC'] else
  'Energy' if t in ['XOM','CVX'] else 'Consumer Staples' for t in tickers}

# ========================== OPTIMIZATION ==========================
opt = ConvexPortfolioOptimizer(tickers, sector_map)
opt.fetch_data(start_date='2018-01-01', end_date='2023-12-31')
opt.compute_moments()

sector_caps = {'Technology': 0.45, 'Financials': 0.30, 'Energy': 0.25, 'Consumer Staples': 0.25}
w_target = opt.optimize(risk_aversion=3.0, position_bounds=(0.01, 0.20), sector_caps=sector_caps)
weights_target = pd.Series(w_target, index=tickers).round(4).sort_values(ascending=False)

print("=== Target Portfolio Weights ===\n", weights_target)


# ========================== WALK-FORWARD BACKTEST ==========================
def simple_walk_forward(returns, optimizer, lookback=756, rebalance_every=21, risk_aversion=3.0):
  returns.index = pd.to_datetime(returns.index)
  dates = returns.index
  rebalance_idx = list(range(lookback, len(dates), rebalance_every))
  weights_df = pd.DataFrame(index=dates, columns=returns.columns, dtype=float)
  
  for idx in rebalance_idx:
    train = returns.iloc[idx-lookback:idx]
    mu_train = train.mean() * 252
    cov_train = LedoitWolf().fit(train).covariance_ * 252
    w = optimizer.optimize(risk_aversion=risk_aversion)  # simplified - uses class moments for demo
    weights_df.loc[dates[idx]] = w
  
  weights_df = weights_df.ffill().fillna(0)
  lagged_weights = weights_df.shift(1).fillna(0)
  port_returns = (lagged_weights * returns).sum(axis=1).dropna()
  
  # Benchmarks
  eq_returns = (returns / returns.shape[1]).sum(axis=1).loc[port_returns.index]
  spy = yf.download('SPY', start=dates[0], end=dates[-1], progress=False)['Close'].pct_change()
  spy = spy.reindex(port_returns.index).fillna(0)
  
  return port_returns, eq_returns, spy

port_returns, eq_returns, spy_returns = simple_walk_forward(opt.returns, opt)

# Build vectorbt portfolios
def build_pf(returns):
  equity = 100_000 * (1 + returns).cumprod()
  return vbt.Portfolio.from_holding(
    close=equity,
    init_cash=100_000,
    freq='D',
  )

pf_opt = build_pf(port_returns)
pf_eq  = build_pf(eq_returns)
pf_spy = build_pf(spy_returns)


# ========================== PORTFOLIO PERFORMANCE PLOT ==========================
fig_perf = make_subplots(rows=2, cols=1, subplot_titles=("Cumulative Returns", "Drawdown"), vertical_spacing=0.12, row_heights=[0.65, 0.35])

for pf, color, name in [(pf_opt, 'royalblue', "Convex Optimizer"), (pf_eq,'orange',"Equal Weight"), (pf_spy, 'green',"SPY")]:
  cum = pf.value() / pf.value().iloc[0]
  fig_perf.add_trace(go.Scatter(x=cum.index, y=cum, name=name, line=dict(color=color)), row=1, col=1)
  fig_perf.add_trace(go.Scatter(x=pf.drawdown().index, y=pf.drawdown(), name=name, line=dict(color=color), fill='tozeroy'), row=2, col=1)

fig_perf.update_layout(height=800, template='plotly_white', title="Portfolio Performance: Convex Optimizer vs Benchmarks")
fig_perf.show()

[*********************100%***********************]  10 of 10 completed


=== Target Portfolio Weights ===
 AAPL     0.20
NVDA     0.20
JPM      0.20
CVX      0.20
PG       0.11
XOM      0.05
MSFT     0.01
GOOGL    0.01
BAC      0.01
KO       0.01
dtype: float64


In [43]:
# ========================== EXECUTION SETUP ==========================
portfolio_value = 10_000_000
prev_weights = np.full(len(tickers), 0.10)
delta_weights = w_target - prev_weights
shares_to_trade = (portfolio_value * delta_weights / 150).astype(int)   # avg price $150

largest_idx = np.argmax(np.abs(shares_to_trade))
X0 = abs(shares_to_trade[largest_idx])
S0 = 150.0
ticker_exec = tickers[largest_idx]

print(f"Executing largest order: {X0:,} shares of {ticker_exec} (Portfolio value = ${portfolio_value:,.0f})")

engine = ExecutionEngineEnhanced(eta=0.00015, gamma=0.00008, sigma=0.018, lambda_risk=1e-5)

volume_profile = np.sin(np.pi * np.linspace(0, 1, 390))**2 + 0.15 * np.exp(-((np.linspace(0,1,390)-0.5)**2)/0.08)
volume_profile /= volume_profile.sum()

# VWAP
vwap_trades = engine.vwap_trajectory(X0, volume_profile)
print("\n=== VWAP Execution ===")
vwap_res = engine.simulate_execution(S0, X0, vwap_trades, strategy_name="Naive VWAP")

# Almgren-Chriss
ac_trades = engine.almgren_chriss_trajectory(X0)
print("\n=== Almgren-Chriss Optimal Execution ===")
ac_res = engine.simulate_execution(S0, X0, ac_trades, strategy_name="Almgren-Chriss")

# Cost Comparison Table
cost_df = pd.DataFrame({
  'Strategy': ['Naive VWAP', 'Almgren-Chriss Optimal'],
  'Implementation Shortfall ($)': [vwap_res['implementation_shortfall'], ac_res['implementation_shortfall']],
  'Avg Slippage per Share ($)': [vwap_res['average_slippage_per_share'], ac_res['average_slippage_per_share']]
})

print("\n=== Execution Cost Comparison ===")
display(cost_df.round(2))

savings = vwap_res['implementation_shortfall'] - ac_res['implementation_shortfall']
print(f"\n💰 Alpha preserved by Almgren-Chriss: **${savings:,.2f}** on this rebalance")

Executing largest order: 6,666 shares of AAPL (Portfolio value = $10,000,000)

=== VWAP Execution ===



=== Almgren-Chriss Optimal Execution ===



=== Execution Cost Comparison ===


,Strategy,Implementation Shortfall ($),Avg Slippage per Share ($)
0,Naive VWAP,6852.56,1.03
1,Almgren-Chriss Optimal,5019.83,0.75



💰 Alpha preserved by Almgren-Chriss: **$1,832.73** on this rebalance
